In [1]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

In [2]:
from typing import Literal,TypedDict

class QuadState(TypedDict):
    a: int
    b: int
    c: int
    equation: str
    discriminant: float
    result: str

In [3]:
def show_equation(state: QuadState):
    equation = f"{state['a']}x^2 + {state['b']}x + {state['c']}"
    return {'equation':equation}

def calculate_discriminant(state: QuadState):

    discr = state['b']**2-(4*state['a']*state['c'])

    return{'discriminant': discr}

In [4]:
def show_result(state: QuadState):
    root1 = (-state["b"] + state["discriminant"]**0.5)/2*state['a']
    root2 = (-state["b"] - state["discriminant"]**0.5)/2*state['a']

    result = f"The roots are {root1} and {root2}"
    return {'result': result}

In [5]:
def no_real_root(state: QuadState):

    root1 = (-state["b"] + state["discriminant"]**0.5)/2*state['a']
    root2 = (-state["b"] - state["discriminant"]**0.5)/2*state['a']

    result = f"No real roots {root1}, {root2}"

    return {'result' : result}

def real_root(state: QuadState):

    root1 = (-state["b"] + state["discriminant"]**0.5)/(2*state['a'])
    root2 = (-state["b"] - state["discriminant"]**0.5)/(2*state['a'])

    result = f"The real roots {root1}, {root2}"

    return {'result' : result}

def repeated_root(state: QuadState):

    state['root1'] = -state['b']+(state['discriminant']**0.5/(2*state['a']))

    result = f"The repeating root is {state['root1'], {state['root2']}}"
    return 

In [9]:
# make conditional routing function

def check_condition(state: QuadState) -> Literal["real_root", "no_real_root", "repeated_root"]:

    if state["discriminant"] >0:
        return "real_root"

    elif state["discriminant"] == 0: 
        return "repeated_root" 

    else:
        return "no_real_root"

In [ ]:
graph = StateGraph(QuadState)

graph.add_node("show_equation", show_equation)
graph.add_node("calculate_discriminant", calculate_discriminant)
graph.add_node("real_root", real_root)
graph.add_node("repeated_root", repeated_root)
graph.add_node("no_real_root", no_real_root)
graph.add_node("show_result", show_result)

graph.add_edge(START, "show_equation")
graph.add_edge("show_equation", "calculate_discriminant")
graph.add_conditional_edges("calculate_discriminant", check_condition)
graph.add_edge("real_root",END)
graph.add_edge("repeated_root", END) 
graph.add_edge("real_root", END)
graph.add_edge("no_real_root", END)


In [12]:
initial_state = {
    "a": 4,
    "b": -3,
    "c": 6
}

workflow = graph.compile()
final_state = workflow.invoke(initial_state)
workflow
print(final_state)

{'a': 4, 'b': -3, 'c': 6, 'equation': '4x^2 + -3x + 6', 'discriminant': -87, 'result': 'No real roots (6.000000000000001+18.65475810617763j), (5.999999999999999-18.65475810617763j)'}
